## Credit deterioration and limit decision

> **Credit:** Is credit quality deteriorating? Is cutting limits by 30% for everyone the right fix? If not, propose something better.

**September shows deterioration among comparable loans. Use repayment history to set limits.** Mature DPD30 rose from 17/260 (6.54%) in July and 25/427 (5.85%) in August to 34/353 (9.63%) in September. Comparing disbursals on days 1-22 gives 5.76%, 5.56% and 9.63%. Finance's ~12% is consistent with 73/623 (11.72%) short at extract, which includes loans that had yet to reach DPD30. September had 270 immature loans. The evidence supports a targeted pilot in place of the proposed blanket 30% cut.

| Repayment outcome among 1,040 mature loans | Count | Proposed action |
|---|---:|---|
| Fully repaid by due date | 658 | Review limits quarterly and pilot increases for consistently on-time borrowers with verified affordability. Include borrowers with verified higher financial capacity who meet the same criteria. |
| Fully repaid after due date, by DPD30 | 306 | Freeze the current limit, pause upgrades, and standardize the reasons for late payment. |
| DPD30 default, later fully repaid | 0 | Standardize cure reasons and test a reduced renewal limit once enough examples are observed. |
| DPD30 default, still unpaid in full at extract | 76 | Suspend new borrowing until full repayment and reassessment. |

**Use the four repayment bands as a simple score.** Start with the latest mature outcome and the share of recent loans paid on time. Review each borrower's repayment consistency quarterly. The Q3 cohort has one loan per borrower, so it cannot validate repeat consistency or identify high-net-worth borrowers.

**Data to request and ROI.** Request repeat loan histories, payment amounts and dates, assigned limits, requested amounts, approval decisions, verified affordability and standardized late-payment and cure reasons. Test the bands against the current policy. Optimize net loan contribution per eligible borrower using collected fees on fully repaid loans less unrecovered principal, funding, servicing and collection costs. Track DPD30 and cure rates alongside contribution.

**Definitions.** Group by disbursal month. Maturity requires due date plus 30 days by November 5. Full repayment means principal plus fee within EGP 0.01. The last payment date proxies for full repayment. Payment on the due date is on time, and full payment on the checkpoint avoids DPD30. Balances still short may cure after extract.


In [1]:
# Task 3.4 | Credit risk and the proposed limit cut
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

ROOT = next(
    folder for folder in (Path.cwd(), Path.cwd().parent)
    if (folder / "data" / "cleaned").is_dir()
)
CLEANED = ROOT / "data" / "cleaned"
Q3_START = "2026-07-01"
Q3_END = "2026-10-01"
EXTRACT_DATE = "2026-11-05"

CHANNEL_MAP = {
    "meta": "Meta", "facebook ads": "Meta", "facebook": "Meta", "fb": "Meta",
    "instagram": "Meta", "ig": "Meta",
    "google": "Google Ads", "google ads": "Google Ads", "adwords": "Google Ads",
    "gads": "Google Ads", "tiktok": "TikTok", "tt": "TikTok",
    "organic": "Other", "(direct)": "Other", "referral": "Other",
    "ref": "Other", "invite": "Other",
}

loans = pd.read_csv(CLEANED / "loans.csv")
users = pd.read_csv(CLEANED / "users.csv", usecols=["user_id", "utm_source_normalized"])
users["channel"] = users["utm_source_normalized"].map(CHANNEL_MAP).fillna("Unknown")
settings = pd.DataFrame([{
    "q3_start": Q3_START, "q3_end": Q3_END,
    "extract_date": EXTRACT_DATE,
}])

engine = create_engine("sqlite://")
loans.to_sql("loans", engine, index=False)
users[["user_id", "channel"]].to_sql("users", engine, index=False)
settings.to_sql("settings", engine, index=False)

%load_ext sql
%config SqlMagic.displaylimit = 20
%sql engine

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [2]:
%%sql
-- 1. Reconcile Finance's snapshot with mature DPD30 by disbursement month.
DROP VIEW IF EXISTS temp.q3_loan_outcomes;

CREATE TEMP VIEW q3_loan_outcomes AS
WITH loan_checkpoints AS (
    SELECT l.*, s.extract_date, DATE(l.due_date, '+30 days') AS dpd30_date,
           STRFTIME('%Y-%m', l.disbursed_date) AS disbursal_month,
           CAST(STRFTIME('%d', l.disbursed_date) AS INTEGER) AS disbursal_day,
           l.principal_egp + l.fee_egp AS amount_due
    FROM loans AS l CROSS JOIN settings AS s
    WHERE l.disbursed_date >= s.q3_start AND l.disbursed_date < s.q3_end
),
repayment_flags AS (
    SELECT *,
           dpd30_date <= extract_date AS is_mature,
           amount_repaid_egp < amount_due - 0.01 AS short_at_extract
    FROM loan_checkpoints
)
SELECT *,
       CASE WHEN is_mature AND (short_at_extract OR last_repayment_date > dpd30_date)
            THEN 1 ELSE 0 END AS default_at_dpd30
FROM repayment_flags;

SELECT disbursal_month, COUNT(*) AS loans_disbursed,
       SUM(short_at_extract) AS loans_short_at_extract,
       ROUND(100.0 * SUM(short_at_extract) / COUNT(*), 2) AS short_at_extract_pct,
       SUM(is_mature) AS mature_loans,
       COUNT(*) - SUM(is_mature) AS immature_loans,
       SUM(default_at_dpd30) AS dpd30_defaults,
       ROUND(100.0 * SUM(default_at_dpd30) / NULLIF(SUM(is_mature), 0), 2) AS dpd30_pct
FROM q3_loan_outcomes
GROUP BY disbursal_month
ORDER BY disbursal_month;

Out[0]: 
+-----------------+-----------------+------------------------+----------------------+--------------+----------------+----------------+-----------+
| disbursal_month | loans_disbursed | loans_short_at_extract | short_at_extract_pct | mature_loans | immature_loans | dpd30_defaults | dpd30_pct |
+-----------------+-----------------+------------------------+----------------------+--------------+----------------+----------------+-----------+
|     2026-07     |       260       |           17           |         6.54         |     260      |       0        |       17       |    6.54   |
|     2026-08     |       427       |           25           |         5.85         |     427      |       0        |       25       |    5.85   |
|     2026-09     |       623       |           73           |        11.72         |     353      |      270       |       34       |    9.63   |
+-----------------+-----------------+------------------------+----------------------+--------------+---------

Running query in 'sqlite://'

In [3]:
%%sql
-- 2. Classify each loan once using the same checkpoints as the monthly rate.
WITH classified AS (
    SELECT disbursal_month,
           CASE
               WHEN NOT is_mature THEN 'Immature'
               WHEN short_at_extract THEN 'Default, still short'
               WHEN last_repayment_date IS NULL THEN 'Missing repayment date'
               WHEN last_repayment_date <= due_date THEN 'Paid on time'
               WHEN last_repayment_date <= dpd30_date THEN 'Paid late by DPD30'
               ELSE 'Default, later paid'
           END AS repayment_outcome
    FROM q3_loan_outcomes
)
SELECT disbursal_month, COUNT(*) AS loans_disbursed,
       SUM(repayment_outcome = 'Paid on time') AS paid_on_time,
       SUM(repayment_outcome = 'Paid late by DPD30') AS paid_late_by_dpd30,
       SUM(repayment_outcome = 'Default, later paid') AS default_later_repaid,
       SUM(repayment_outcome = 'Default, still short') AS default_still_short,
       SUM(repayment_outcome = 'Immature') AS immature,
       SUM(repayment_outcome = 'Missing repayment date') AS missing_repayment_date
FROM classified
GROUP BY disbursal_month
ORDER BY disbursal_month;

Out[0]: 
+-----------------+-----------------+--------------+--------------------+----------------------+---------------------+----------+------------------------+
| disbursal_month | loans_disbursed | paid_on_time | paid_late_by_dpd30 | default_later_repaid | default_still_short | immature | missing_repayment_date |
+-----------------+-----------------+--------------+--------------------+----------------------+---------------------+----------+------------------------+
|     2026-07     |       260       |     176      |         67         |          0           |          17         |    0     |           0            |
|     2026-08     |       427       |     277      |        125         |          0           |          25         |    0     |           0            |
|     2026-09     |       623       |     205      |        114         |          0           |          34         |   270    |           0            |
+-----------------+-----------------+--------------+---------

Running query in 'sqlite://'

## Appendix | Matched-period and borrower-mix checks

In [4]:
%%sql
-- A1. Compare the same days of each month, using September's maturity cutoff.
WITH comparable_window AS (
    SELECT CAST(STRFTIME('%d', DATE(extract_date, '-44 days')) AS INTEGER) AS last_day
    FROM settings
)
SELECT disbursal_month, last_day AS through_day, COUNT(*) AS mature_loans,
       SUM(default_at_dpd30) AS dpd30_defaults,
       ROUND(100.0 * AVG(default_at_dpd30), 2) AS dpd30_pct
FROM q3_loan_outcomes CROSS JOIN comparable_window
WHERE is_mature AND disbursal_day <= last_day
GROUP BY disbursal_month, last_day
ORDER BY disbursal_month;

Out[0]: 
+-----------------+-------------+--------------+----------------+-----------+
| disbursal_month | through_day | mature_loans | dpd30_defaults | dpd30_pct |
+-----------------+-------------+--------------+----------------+-----------+
|     2026-07     |      22     |     139      |       8        |    5.76   |
|     2026-08     |      22     |     288      |       16       |    5.56   |
|     2026-09     |      22     |     353      |       34       |    9.63   |
+-----------------+-------------+--------------+----------------+-----------+


Running query in 'sqlite://'

In [5]:
%%sql
-- A2. Compare mature default rates by booked amount and acquisition channel.
WITH mature_loans AS (
    SELECT q.disbursal_month, q.principal_egp,
           COALESCE(u.channel, 'Unknown') AS channel, q.default_at_dpd30
    FROM q3_loan_outcomes AS q
    LEFT JOIN users AS u USING (user_id)
    WHERE q.is_mature
),
segments AS (
    SELECT 'Booked principal (EGP)' AS breakdown,
           CAST(principal_egp AS TEXT) AS segment, disbursal_month, default_at_dpd30
    FROM mature_loans
    UNION ALL
    SELECT 'Acquisition channel', channel, disbursal_month, default_at_dpd30
    FROM mature_loans
)
SELECT breakdown, segment,
       COUNT(*) FILTER (WHERE disbursal_month = '2026-07') AS jul_mature,
       SUM(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-07') AS jul_defaults,
       ROUND(100.0 * AVG(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-07'), 2) AS jul_pct,
       COUNT(*) FILTER (WHERE disbursal_month = '2026-08') AS aug_mature,
       SUM(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-08') AS aug_defaults,
       ROUND(100.0 * AVG(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-08'), 2) AS aug_pct,
       COUNT(*) FILTER (WHERE disbursal_month = '2026-09') AS sep_mature,
       SUM(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-09') AS sep_defaults,
       ROUND(100.0 * AVG(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-09'), 2) AS sep_pct
FROM segments
GROUP BY breakdown, segment
ORDER BY breakdown DESC, CAST(segment AS INTEGER), segment;

Out[0]: 
+------------------------+------------+------------+--------------+---------+------------+--------------+---------+------------+--------------+---------+
|       breakdown        |  segment   | jul_mature | jul_defaults | jul_pct | aug_mature | aug_defaults | aug_pct | sep_mature | sep_defaults | sep_pct |
+------------------------+------------+------------+--------------+---------+------------+--------------+---------+------------+--------------+---------+
| Booked principal (EGP) |    500     |     63     |      0       |   0.0   |    116     |      4       |   3.45  |    100     |      11      |   11.0  |
| Booked principal (EGP) |    1000    |     81     |      5       |   6.17  |    132     |      7       |   5.3   |     94     |      11      |   11.7  |
| Booked principal (EGP) |    1500    |     56     |      5       |   8.93  |     78     |      10      |  12.82  |     82     |      9       |  10.98  |
| Booked principal (EGP) |    2000    |     35     |      4       |

Running query in 'sqlite://'